# AO3 Tags-per-Story Statistics

Descriptive statistics on how many distinct tags each work carries, from
`ao3_tag_metadata.csv` -- both pooled across all seven tag-bearing fields
(`rating`, `warnings`, `category`, `fandom`, `relationship`, `character`,
`additional_tags`) and broken down per field.

Handles the dataset's shape quirks: the scraper emits one row per (seed tag,
work) so a work found via several tags is deduped to one; comma-separated cell
values are deduped within the cell; a tag is namespaced `field::value` so the
same string in two fields counts twice. Zeros are included -- a story with no
`additional_tags` counts as 0, so the per-field means are over every story.

No network access is required -- this only reads a local CSV.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present.
# Safe to re-run.
%pip install -q pandas

In [ ]:
import pandas as pd

## Configuration

In [ ]:
INPUT = "ao3_tag_metadata.csv"
OUT = "ao3_tags_per_story_stats.csv"

DELIMITER = ", "
ALL_METADATA_FIELDS = ["rating", "warnings", "category", "fandom",
                        "relationship", "character", "additional_tags"]

## Load metadata

In [ ]:
# copied from ao3_tag_visualizer.py -- keep in sync if that file changes
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    values = [v.strip() for v in cell.split(delimiter) if v.strip()]
    return list(dict.fromkeys(values))


def explode_field(df, field):
    exploded = df[["tag", "work_id", field]].copy()
    exploded[field] = exploded[field].map(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


def build_document_tag_table(df, fields=ALL_METADATA_FIELDS):
    deduped = df.drop_duplicates(subset="work_id", keep="first")
    tables = []
    for field in fields:
        exploded = explode_field(deduped, field)
        exploded = exploded.rename(columns={field: "value"})
        exploded["tag_id"] = field + "::" + exploded["value"]
        tables.append(exploded[["work_id", "tag_id"]])
    return pd.concat(tables, ignore_index=True)


df = load_metadata(INPUT)
df.head()

## Tags-per-story statistics

In [ ]:
# copied from ao3_tag_counts.py -- keep in sync if that file changes
STAT_COLUMNS = ["scope", "n_works", "total_tags", "mean", "std",
                "min", "p25", "median", "p75", "max"]


def tags_per_story_stats(df):
    """Returns a DataFrame [scope, n_works, total_tags, mean, std, min,
    p25, median, p75, max], one row per scope: "all_fields" (every tag
    field pooled) followed by one row per field in ALL_METADATA_FIELDS.

    The denominator is every distinct work in df; a work with no tags in a
    given scope contributes 0 to that scope (zeros included), so the means
    describe every story, not just the ones that happen to use the field."""
    all_work_ids = pd.Index(df["work_id"].unique(), name="work_id")

    table = build_document_tag_table(df, fields=ALL_METADATA_FIELDS)
    table = table.assign(field=table["tag_id"].str.split("::", n=1).str[0])

    def stats_for(scope, counts_per_work):
        full = counts_per_work.reindex(all_work_ids, fill_value=0)
        desc = full.describe()  # count, mean, std, min, 25%, 50%, 75%, max
        std = desc["std"]
        return {
            "scope": scope,
            "n_works": int(desc["count"]),
            "total_tags": int(full.sum()),
            "mean": round(float(desc["mean"]), 2),
            # std is NaN when there's a single work (pandas ddof=1); report 0.
            "std": round(float(std), 2) if pd.notna(std) else 0.0,
            "min": int(desc["min"]),
            "p25": round(float(desc["25%"]), 1),
            "median": round(float(desc["50%"]), 1),
            "p75": round(float(desc["75%"]), 1),
            "max": int(desc["max"]),
        }

    rows = [stats_for("all_fields", table.groupby("work_id").size())]
    for field in ALL_METADATA_FIELDS:
        field_counts = table[table["field"] == field].groupby("work_id").size()
        rows.append(stats_for(field, field_counts))

    return pd.DataFrame(rows, columns=STAT_COLUMNS)


stats = tags_per_story_stats(df)
stats.to_csv(OUT, index=False)
print(f"wrote {OUT} ({stats.iloc[0]['n_works']} works)")
stats

## Done

`{OUT}` is now in the notebook's working directory: one row for the pooled
all-fields total plus one per field, each with count / mean / std / min /
quartiles / max. Re-run from the Configuration cell with a different `INPUT`
to reanalyze.